In [14]:
from db_init import init_rag_db

client, collections = init_rag_db()

Collection 'audios' ready.
Collection 'audio_segments' ready.
Collection 'transcribe_segments' ready.
Collection 'audio_attributes' ready.


In [15]:
from audio_processor import process_audio

# Define target input file
audio_file = "./data/Oral History Audio Interviews/AbigailBThomas_FINAL_1_.wav"

process_audio(audio_file)

Using cache found in /Users/krupali/.cache/torch/hub/snakers4_silero-vad_master
/Users/krupali/.virtualenvs/.venv/lib/python3.14/site-packages/torch/jit/_serialization.py:170: FutureWarning: `torch.jit.load` is not supported in Python 3.14+ and may break. Please switch to `torch.export`.
  warnings.warn(


--- Processing Root Audio: AbigailBThomas_FINAL_1_.wav (ID: audio_bc6f7d20) ---
Running diarization...
Running VAD...
Splitting by silence found 189 utterances
Extracting embeddings...


Utterances: 100%|██████████| 189/189 [00:02<00:00, 63.17it/s]
/Users/krupali/.virtualenvs/.venv/lib/python3.14/site-packages/sklearn/manifold/_spectral_embedding.py:305: UserWarning: Array is not symmetric, and will be converted to symmetric by average with its transpose.
  adjacency = check_symmetric(adjacency)


Clustering to 2 speakers...
Cleaning up output...
Done!
Extracted 18 merged segments. Processing cuts and transcriptions...
Transcription error for ./data/segments/audio_bc6f7d20_seg_0.wav: headers: {'date': 'Sat, 26 Sep 2026 08:49:11 GMT', 'content-type': 'application/json', 'content-length': '220', 'connection': 'keep-alive', 'server': 'uvicorn', 'x-request-id': '20260926_2b43b0ba-75ce-4333-9546-b25fa5e5ace8', 'access-control-allow-origin': '*', 'strict-transport-security': 'max-age=31536000; includeSubDomains', 'x-frame-options': 'DENY', 'x-content-type-options': 'nosniff', 'referrer-policy': 'strict-origin-when-cross-origin', 'permissions-policy': 'geolocation=(), camera=(), microphone=(), payment=()', 'content-security-policy': "default-src 'none'; frame-ancestors 'none'", 'cache-control': 'no-store, no-cache, must-revalidate', 'pragma': 'no-cache', 'expires': '0', 'x-xss-protection': '0'}, status_code: 400, body: {'error': {'message': 'Audio duration exceeds the maximum limit of 

In [2]:
from query_helper import AudioRAGQuerier

In [16]:
from query_helper import AudioRAGQuerier

rag = AudioRAGQuerier()

# --- Example 1: Semantic Search ---
print("=== 1. Semantic Search ===")
search_results = rag.semantic_search(query_text="where were you growing up", n_results=3)
for r in search_results:
    print(f"[{r['start_sec']}s - {r['end_sec']}s] Speaker {r['speaker_label']}: {r['transcript']}")
    print(f" Audio File: {r['audio_blob']}\n")


=== 1. Semantic Search ===
[288.674s - 292.4139375s] Speaker 1: How was it growing up here in in in New York?
 Audio File: None

[186.594s - 188.551s] Speaker 1: When and I mean where were you?
 Audio File: None

[210.594s - 213.557s] Speaker 1: Have you lived all your life in New York?
 Audio File: None



In [5]:

# --- Example 2: Chronological Full Transcript ---
print("=== 2. Ordered Transcript ===")
ordered_transcript = rag.get_ordered_segments()
print(f"Total Segments: {len(ordered_transcript)}")
for seg in ordered_transcript[:5]:  # Print first 5
    print(f"[{seg['start_sec']:.1f}s - {seg['end_sec']:.1f}s] Speaker {seg['speaker_label']}: {seg['text']}")


=== 2. Ordered Transcript ===
Total Segments: 12


KeyError: 'speaker_label'

In [7]:
rag = AudioRAGQuerier()

print("--- Time Range Query (0.0s to 120.0s) ---")
hits = rag.get_segments_by_time_range(min_sec=0.0, max_sec=32.0)
print(f"Total matching segments: {len(hits)}")

for h in hits[:5]:
    print(f"[{h['start_sec']:.1f}s - {h['end_sec']:.1f}s] Speaker {h['speaker']}: {h['text']}")

--- Time Range Query (0.0s to 120.0s) ---
Total matching segments: 1
[31.3s - 42.0s] Speaker 0: Midtown Manhattan. Okay, so hi Abigail. Hi Natalie, how are you doing? Good, how are you? I'm doing pretty good. Okay, great. So.


In [9]:

# --- Example 4: Speaker-Specific Dialogue ---
print("\n=== 4. Speaker 1 Dialogue ===")
spk1_lines = rag.get_speaker_dialogue(speaker_label=1)
for line in spk1_lines[:5]:
    print(f"[{line['start_sec']:.1f}s - {line['end_sec']:.1f}s] {line['text']}")


=== 4. Speaker 1 Dialogue ===
[42.0s - 45.8s] Tell me, oh tell us your name, tell me your name and your age.
[75.8s - 78.4s] And how would you describe your gender?
[136.9s - 144.6s] Tell me a little bit more about your daily life in terms of being non-binary, like being you and and your.
[186.6s - 188.6s] When and I mean where were you?
[210.6s - 213.6s] Have you lived all your life in New York?


In [3]:
search_results = rag.semantic_search(query_text="hospital", n_results=3)
for r in search_results:
    print(search_results)
    print(f"[{r['start_sec']}s - {r['end_sec']}s] Speaker {r['speaker_label']}: {r['transcript']}")
    print(f" Audio File: {r['audio_blob']}\n")

[{'segment_id': 'audio_bc6f7d20_seg_8', 'transcript': 'When and I mean where were you?', 'distance': 0.6198410987854004, 'speaker_label': 1, 'start_sec': 186.594, 'end_sec': 188.551, 'audio_blob': None}, {'segment_id': 'audio_bc6f7d20_seg_9', 'transcript': "I was originally born in Brooklyn, New York, over at Victor Memorial Hospital. Actually, only part of it remains. I think the other part was sold for condominiums. Okay, and so I'm old enough to know what that feels like now. Okay. Fair enough. So, um.", 'distance': 0.6611588597297668, 'speaker_label': 0, 'start_sec': 188.551, 'end_sec': 210.3979375, 'audio_blob': None}, {'segment_id': 'audio_bc6f7d20_seg_4', 'transcript': 'And how would you describe your gender?', 'distance': 0.6722559332847595, 'speaker_label': 1, 'start_sec': 75.778, 'end_sec': 78.4299375, 'audio_blob': None}]
[186.594s - 188.551s] Speaker 1: When and I mean where were you?
 Audio File: None

[{'segment_id': 'audio_bc6f7d20_seg_8', 'transcript': 'When and I mean 

In [4]:
import chromadb

client = chromadb.PersistentClient(path="./rag_store")
attr_col = client.get_collection("audio_attributes")

print("Total in audio_attributes:", attr_col.count())
data = attr_col.get(limit=5)
for i in range(len(data['ids'])):
    print(f"ID: {data['ids'][i]}")
    print(f"Meta: {data['metadatas'][i]}\n")

Total in audio_attributes: 1
ID: attr_audio_bc6f7d20
Meta: {'channels': 1, 'frame_rate': 16000, 'frame_width': 2, 'audio_id': 'audio_bc6f7d20', 'sample_width': 2}



In [9]:
import chromadb

client = chromadb.PersistentClient(path="./rag_store")
attrs = client.get_collection("audio_attributes").get(limit=3)

for i, meta in enumerate(attrs['metadatas']):
    print(f"--- Record {i+1} ---")
    print(meta)

--- Record 1 ---
{'channels': 1, 'frame_width': 2, 'sample_width': 2, 'audio_id': 'audio_bc6f7d20', 'frame_rate': 16000}


In [10]:
%pip install rank-bm25

Note: you may need to restart the kernel to use updated packages.


In [31]:
from rank_bm25 import BM25Okapi


def hybrid_search(query: str, collections: dict, top_k: int = 5, k: int = 60):
    """
    Simple hybrid search with guaranteed attribute resolution derived directly from trans_id.
    """
    trans_col = collections["transcribe_segments"]
    attr_col = collections["audio_attributes"]
    seg_col = collections["audio_segments"]

    # 1. Vector Search (Semantic)
    query_res = trans_col.query(query_texts=[query], n_results=top_k)
    vector_ids = query_res["ids"][0] if query_res["ids"] else []

    # 2. Keyword Search (BM25)
    all_data = trans_col.get()
    if not all_data["documents"]:
        return []

    bm25 = BM25Okapi([doc.lower().split() for doc in all_data["documents"]])
    bm25_scores = bm25.get_scores(query.lower().split())

    keyword_ids = [
        all_data["ids"][i]
        for i in sorted(range(len(bm25_scores)), key=lambda i: bm25_scores[i], reverse=True)[:top_k]
    ]

    # 3. Simple RRF Scoring
    rrf_scores = {}

    for rank, doc_id in enumerate(vector_ids, start=1):
        rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + (1.0 / (k + rank))

    for rank, doc_id in enumerate(keyword_ids, start=1):
        rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + (1.0 / (k + rank))

    # Sort top items by RRF score
    fused_results = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)[:top_k]

    # 4. Fetch Details & Format Output
    results = []
    for trans_id, score in fused_results:
        # Get transcript text
        trans_data = trans_col.get(ids=[trans_id])
        if not trans_data["ids"]:
            continue

        transcript_text = trans_data["documents"][0]
        trans_meta = trans_data["metadatas"][0] if trans_data["metadatas"] and trans_data["metadatas"][0] else {}

        # DIRECT ID DERIVATION (Guarantees matching across collections)
        # "trans_audio_123_seg_0" -> "attr_audio_123_seg_0"
        # "trans_audio_123_seg_0" -> "audio_123_seg_0"
        attr_id = trans_id.replace("trans_", "attr_", 1)
        seg_id = trans_id.replace("trans_", "", 1)

        # Query audio_attributes safely
        attr_data = attr_col.get(ids=[attr_id])
        attr = (
            attr_data["metadatas"][0]
            if attr_data["metadatas"] and attr_data["metadatas"][0]
            else {}
        )

        # Query audio_segments safely
        seg_data = seg_col.get(ids=[seg_id])
        seg = (
            seg_data["metadatas"][0]
            if seg_data["metadatas"] and seg_data["metadatas"][0]
            else {}
        )

        # Extract with multi-level fallbacks (attr_col -> trans_meta -> default)
        start_sec = attr.get("start_sec") if "start_sec" in attr else trans_meta.get("start_sec")
        end_sec = attr.get("end_sec") if "end_sec" in attr else trans_meta.get("end_sec")
        speaker = attr.get("speaker_label", trans_meta.get("speaker_label", "Unknown"))
        blob = seg.get("blob_path", trans_meta.get("blob_path", "N/A"))

        results.append({
            "transcript": transcript_text,
            "rrf_score": score,
            "start_sec": start_sec,
            "end_sec": end_sec,
            "speaker_label": speaker,
            "audio_blob": blob
        })

    return results

In [18]:
from audio_processor import init_rag_db

# 1. Initialize DB and get the collections dictionary
client, collections = init_rag_db()


In [ ]:
# Execute Hybrid Search
query = "What did they discuss regarding oral history records?"
# query = "New york"
# query = "Miami"
matches = hybrid_search(query, collections, top_k=5)

# Print combined results safely
for m in matches:
    start_sec = m.get('start_sec')
    end_sec = m.get('end_sec')
    
    # Safe float formatting guard
    start_str = f"{start_sec:.1f}s" if isinstance(start_sec, (int, float)) else "N/A"
    end_str = f"{end_sec:.1f}s" if isinstance(end_sec, (int, float)) else "N/A"
    speaker = m.get('speaker_label', 'Unknown')
    rrf_score = m.get('rrf_score', 0.0)

    print(f"[{start_str} - {end_str}] Speaker {speaker} (RRF Score: {rrf_score:.4f}):")
    print(f" Transcript: {m['transcript']}")
    print(f" Audio Slice: {m['audio_blob']}\n")

[451.8s - 462.2s] Speaker 1 (RRF Score: 0.0164):
 Transcript: Remember any great stories or legends, either in New York or in or in Miami, that you can recall.
 Audio Slice: N/A

[31.3s - 42.0s] Speaker 0 (RRF Score: 0.0164):
 Transcript: Midtown Manhattan. Okay, so hi Abigail. Hi Natalie, how are you doing? Good, how are you? I'm doing pretty good. Okay, great. So.
 Audio Slice: N/A

[186.6s - 188.6s] Speaker 1 (RRF Score: 0.0161):
 Transcript: When and I mean where were you?
 Audio Slice: N/A

[42.0s - 45.8s] Speaker 1 (RRF Score: 0.0161):
 Transcript: Tell me, oh tell us your name, tell me your name and your age.
 Audio Slice: N/A

[288.7s - 292.4s] Speaker 1 (RRF Score: 0.0159):
 Transcript: How was it growing up here in in in New York?
 Audio Slice: N/A



In [33]:
matches[0]

{'transcript': 'Tell me, oh tell us your name, tell me your name and your age.',
 'rrf_score': 0.03149801587301587,
 'start_sec': 42.023,
 'end_sec': 45.773,
 'speaker_label': 1,
 'audio_blob': 'N/A'}